In [1]:
from pathlib import Path

from ovito.io import import_file, export_file
from ovito.modifiers import CreateBondsModifier, CoordinationAnalysisModifier, ExpressionSelectionModifier, InvertSelectionModifier, DeleteSelectedModifier
import numpy as np

In [ ]:
file_path = '/Users/loganyamamoto/Desktop/Research/grants/geo_sciences/bubble_collapse/data/systems/sioh_cov/a-SiO/2_surface/N113.4M/0004.data'
_path = Path(file_path)
shifted_path = _path.parent / f"{_path.stem}_shifted{_path.suffix}"
pipeline = import_file(file_path)
temp = 300
surface_dist = 1.2  # Å from top/bottom surfaces
bond_cutoff = 1.2
refit_box = True  # Shift atoms so min position is 0 and resize box to atom extent

# Non-periodic direction: 'x', 'y', or 'z' (slab normal / confined axis)
direction = 'z'
axis = {'x': 0, 'y': 1, 'z': 2}[direction]

In [3]:
# Ensure correct PBC: non-periodic along chosen direction, periodic in the other two
def set_pbc(frame, data):
    cell = data.cell_  # Use mutable version with underscore
    pbc = [True, True, True]
    pbc[axis] = False
    cell.pbc = tuple(pbc)

pipeline.modifiers.append(set_pbc)

In [4]:
# Create the modifier with Pairwise mode
bond_modifier = CreateBondsModifier(
    mode = CreateBondsModifier.Mode.Pairwise
)

# Set pairwise cutoffs using the set_pairwise_cutoff method
# Note: (1, 2) and (2, 1) are the same pair, so only need to set once
bond_modifier.set_pairwise_cutoff(1, 2, bond_cutoff)

pipeline.modifiers.append(bond_modifier)

In [5]:
# Coordination analysis based on existing bonds
# This modifier counts the number of bonds for each particle
def compute_coordination_from_bonds(frame, data):
    """Compute coordination number from existing bonds."""
    # Get bonds
    bonds = data.particles.bonds
    if bonds is None:
        raise RuntimeError("No bonds found. Make sure CreateBondsModifier runs before this.")
    
    # Get bond topology array (each row contains [particle1_index, particle2_index])
    topology = bonds.topology.array
    
    # Initialize coordination array
    num_particles = data.particles.count
    coordination = np.zeros(num_particles, dtype=int)
    
    # Count bonds for each particle
    # topology[:, 0] contains first particle indices, topology[:, 1] contains second particle indices
    np.add.at(coordination, topology[:, 0], 1)  # Increment coordination for first particles
    np.add.at(coordination, topology[:, 1], 1)  # Increment coordination for second particles
    
    # Store as particle property
    data.particles_.create_property('Coordination', data=coordination)

pipeline.modifiers.append(compute_coordination_from_bonds)

In [6]:
# Function to export selected subset (adds modifiers, exports, then removes them)
def export_subset(mask, filename):
    mask_copy = mask.copy()
    def select_subset(frame, data):
        data.particles_.create_property("Selection", data=mask_copy.astype(int))
    pipeline.modifiers.append(select_subset)
    pipeline.modifiers.append(InvertSelectionModifier())
    pipeline.modifiers.append(DeleteSelectedModifier())
    try:
        export_file(pipeline, filename, "lammps/dump",
                    columns = ["Particle Identifier", "Particle Type", "Position.X", "Position.Y", "Position.Z"])
    finally:
        for _ in range(3):
            pipeline.modifiers.pop()

# ---- Pass 1: compute bonds and coordination (no shifts yet) ----
data = pipeline.compute()

positions = data.particles.positions
coord = data.particles['Coordination'].array
axis_coords = positions[:, axis]
# Ovito cell: columns 0,1,2 = cell vectors a,b,c; column 3 = origin. Bounds = origin to origin + diagonal.
axis_min = data.cell[axis, 3]
axis_max = data.cell[axis, 3] + data.cell[axis, axis]
print(f"{direction}_min, {direction}_max:", axis_min, axis_max)
axis_mid = 0.5 * (axis_max + axis_min)

axis_box_size = axis_max - axis_min

free_mask = (coord == 0)

# Bottom surface: free atoms in bottom zone (select and move these first)
bottom_mask = free_mask & (axis_coords <= axis_mid) & (axis_coords <= axis_min + surface_dist)

# Export bottom free atoms and their IDs (from current state, before any shift)
export_subset(bottom_mask, "free_atoms_bottom.dump")
particle_ids = data.particles['Particle Identifier'].array
bottom_ids = particle_ids[bottom_mask]
with open("free_atoms_bottom_ids.txt", "w") as f:
    f.write(" || ".join(f"ParticleIdentifier == {pid}" for pid in bottom_ids))
print("Pass 1: saved bottom free atoms (before shifting).")

# ---- Shift bottom free atoms by +axis_box (so bonds are recomputed without them) ----
bottom_mask_copy = bottom_mask.copy()
def shift_bottom_along_axis(frame, data):
    pos = np.copy(data.particles.positions)
    pos[bottom_mask_copy, axis] += axis_box_size
    data.particles_.positions_ = pos
pipeline.modifiers.append(shift_bottom_along_axis)

# ---- Pass 2: recompute bonds and coordination after moving bottom ----
bond_modifier_2 = CreateBondsModifier(mode=CreateBondsModifier.Mode.Pairwise)
bond_modifier_2.set_pairwise_cutoff(1, 2, 1.2)
pipeline.modifiers.append(bond_modifier_2)
pipeline.modifiers.append(compute_coordination_from_bonds)

data2 = pipeline.compute()

# ---- Top surface: free atoms in top zone (re-checked after bottom move) ----
coord2 = data2.particles['Coordination'].array
positions2 = data2.particles.positions
axis_coords2 = positions2[:, axis]
free_mask2 = (coord2 == 0)
top_mask = free_mask2 & (axis_coords2 > axis_mid) & (axis_coords2 >= axis_max - surface_dist)

# Export top free atoms and their IDs (after bottom was moved)
export_subset(top_mask, "free_atoms_top.dump")
top_ids = data2.particles['Particle Identifier'].array[top_mask]
with open("free_atoms_top_ids.txt", "w") as f:
    f.write(" || ".join(f"ParticleIdentifier == {pid}" for pid in top_ids))
print("Pass 2: re-checked top surface and saved top free atoms.")
print("Saved particle ID filter strings to free_atoms_top_ids.txt and free_atoms_bottom_ids.txt")



z_min, z_max: 0.5197082633714 406.16112819594804
Pass 1: saved bottom free atoms (before shifting).
Pass 2: re-checked top surface and saved top free atoms.
Saved particle ID filter strings to free_atoms_top_ids.txt and free_atoms_bottom_ids.txt


In [7]:
# Shift top free atoms by -axis_box and export full structure (bottom already shifted in cell 5)
top_mask_copy = top_mask.copy()

def shift_top_along_axis(frame, data):
    pos = np.copy(data.particles.positions)
    pos[top_mask_copy, axis] -= axis_box_size
    data.particles_.positions_ = pos

def shift_and_resize_box(frame, data):
    pos = np.copy(data.particles.positions)
    axis_min_val = pos[:, axis].min()
    pos[:, axis] -= axis_min_val
    axis_range = pos[:, axis].max()
    data.particles_.positions_ = pos

    cell = data.cell_
    cell[axis, 3] = 0.0
    cell[axis, axis] = axis_range

pipeline.modifiers.append(shift_top_along_axis)
n_extra = 1
if refit_box:
    pipeline.modifiers.append(shift_and_resize_box)
    n_extra += 1
export_file(pipeline, str(shifted_path), "lammps/data")
for _ in range(n_extra):
    pipeline.modifiers.pop()
print(f"Saved modified structure to {shifted_path}")

Saved modified structure to /Users/loganyamamoto/Desktop/Research/grants/geo_sciences/bubble_collapse/data/systems/sioh_cov/a-SiO/2_surface/N113.4M/0003_shifted.data


In [8]:
# Verification: counts and pipeline state (run after cells 5 and 6)
assert bottom_mask.sum() == len(bottom_ids), "Bottom mask count vs saved IDs"
assert top_mask.sum() == len(top_ids), "Top mask count vs saved IDs"
print(f"Bottom free atoms: {bottom_mask.sum()} (IDs saved: {len(bottom_ids)})")
print(f"Top free atoms (after re-check): {top_mask.sum()} (IDs saved: {len(top_ids)})")
print(f"Pipeline modifiers after cell 6: {len(pipeline.modifiers)} (expected 6: pbc, bonds, coord, shift_bottom_along_axis, bonds2, coord2)")

Bottom free atoms: 0 (IDs saved: 0)
Top free atoms (after re-check): 0 (IDs saved: 0)
Pipeline modifiers after cell 6: 6 (expected 6: pbc, bonds, coord, shift_bottom_along_axis, bonds2, coord2)


In [9]:
# Load the exported shifted structure and verify positions and box bounds
shifted_pipeline = import_file(str(shifted_path))
data_shifted = shifted_pipeline.compute()
axis_all = data_shifted.particles.positions[:, axis]
box_origin = data_shifted.cell[axis, 3]
box_size = data_shifted.cell[axis, axis]
print(f"Exported file: {shifted_path!s}")
print(f"Minimum {direction} (all particles): {axis_all.min()}")
print(f"Maximum {direction} (all particles): {axis_all.max()}")
print(f"Box {direction} origin: {box_origin}, box {direction} size: {box_size}")

Exported file: /Users/loganyamamoto/Desktop/Research/grants/geo_sciences/bubble_collapse/data/systems/sioh_cov/a-SiO/2_surface/N113.4M/0003_shifted.data
Minimum z (all particles): 0.0
Maximum z (all particles): 405.64138407
Box z origin: 0.0, box z size: 405.64138407


In [10]:
axis_length = axis_all.max() - axis_all.min()
print(f"Length of {direction} axis: {axis_length}")

Length of z axis: 405.64138407
